# Free Energy Calculation of Deca-alanine via Umbrella Sampling

![Graphical Abstract](./assets/figures/puling_deca_alanine.gif)

This example introduces a workflow for calculating the free energy profile of the helix-coil transition (extension process) of **Deca-alanine (10 alanine residues)**, a typical biomolecular model, using **Umbrella Sampling**.

Specifically, we use molecular dynamics simulations to sample structural changes by pulling both ends of the peptide and then calculate the free energy using the MBAR method.

## Notebook Overview

The calculation is divided into the following six steps (notebooks).
By executing them in order, you can perform the entire process from modeling to analysis. Note that inputs required for each notebook are provided in the `assets` directory, allowing them to be run independently.


| Step | Notebook | 概要 |
| :--- | :--- | :--- |
| **01** | [01_modeling_peptide_en.ipynb](./01_modeling_peptide_en.ipynb) | **Modeling**<br>Create the initial structure (α-helix) using AmberTools' `tleap`. |
| **02** | [02_equilibrium_nvt_md_en.ipynb](./02_equilibrium_nvt_md_en.ipynb) | **Equilibration MD**<br>Relax the created structure under the PFP potential to obtain a stable initial structure |
| **03** | [03_steered_md_en.ipynb](./03_steered_md_en.ipynb) | **Steered MD (SMD)**<br> Use PLUMED to gradually pll (or compress) both ends of the peptide to create a trajectory covering the entire reaction coordinate|
| **04** | [04_select_umbrella_sampling_initial_structures_en.ipynb](./04_select_umbrella_sampling_initial_structures_en.ipynb) | **Select Initial Structure**<br>Extract structures corresponding to each umbrella sampling window (specific values of the reaction coordinate) from the SMD trajectory. |
| **05** | [05_umbrella_sampling_en.ipynb](./05_umbrella_sampling_en.ipynb) | **Umbrella Sampling**<br>Using the extracted structures as starting points, run multiple MD simulations in parallel constrained by harmonic potentials to collect data. |
| **06** | [06_mbar_free_energy_en.ipynb](./06_mbar_free_energy_en.ipynb) | **Free Energy Analysis**<br>Analyze the collected data using the MBAR method (`pymbar`) and plot the Potential of Mean Force (PMF) against the end-to-end distance. |

# Target System and Reaction Coordinate

* **Molecule**: Deca-alanine (Ace-Ala10-Nme)
  * Both ends are capped with an acetyl group (ACE) and an N-methyl group (NME).T
  * The simulation is performed in a vacuum.

* **Reaction Coordinate (Collective Variable)**:
  * The distance between both ends of the peptide (specifically the $C\alpha$ atoms near the N-terminus and C-terminus).
  * This distance is varied from approximately from 13.0 Å to 32.0 Å.

## Required Libraries and Environment Setup

The following software packages are required to run this example:

* **PFP-API-CLIENT**: Package for using the Preferred Potential (PFP).
* **ASE (Atomic Simulation Environment)**: Used for atomic structure manipulation and as an MD interface.
* **AmberTools**: Used for initial structure creation in Step 01 (`tleap` command).
* **PLUMED**: Used for adding harmonic oscillator constraints in Steered MD and Umbrella Sampling (Steps 03 and 05).
  * The ASE-PLUMED Calculator interface is used.
* **pymbar**: Used for free energy analysis (MBAR method) in Step 06.


### How to Install PLUMED

[PLUMED](https://www.plumed.org/) is an open-source library that works with molecular dynamics packages to calculate collective variables and apply biasing forces.
This enables efficient sampling of rare events and free energy analysis.
Since this notebook uses PLUMED to add harmonic oscillators to the reaction coordinate (end-to-end distance), you must install it beforehand.

The installation steps for PLUMED are as follows:

#### 1. Download and Compile Source Code

Execute the following commands in your terminal to build PLUMED:

```bash
# Directory for installation
mkdir -p ~/local && cd ~/local

# Clone the PLUMED repository
git clone https://github.com/plumed/plumed2.git plumed-2.9.0

# Checkout version v2.9.0
cd plumed-2.9.0 && git checkout v2.9.0

# Build PLUMED (configure & make)
./configure --disable-mpi --prefix=$HOME/local/plumed-2.9.0 && make -j"$(nproc)" && make install
```

#### 2. Install Python Bindings
Install the Python package required to call PLUMED from ASE:

```bash
$ pip install plumed
```


#### Note:
At the beginning of notebooks [03_steered_md_en.ipynb](./03_steered_md_en.ipynb) and [05_umbrella_sampling_en.ipynb](./05_umbrella_sampling_en.ipynb), there is a cell to specify the PLUMED installation path (e.g., `~/local/plumed-2.9.0`).
Please modify this path according to your installed version or directory name before running.